# AIS-History

### Data Parser from JSON format

In [20]:
import orjson
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime, timezone, timedelta
import pyais
from tqdm import tqdm
import os
import polars as pl
import folium
import math
from folium.plugins import TimestampedGeoJson
from pathlib import Path

In [6]:
CANONICAL_SCHEMA = {
        "class": pa.string(),
        "device": pa.string(),
        "version": pa.int64(),
        "driver": pa.int64(),
        "hardware": pa.string(),
        "rxtime": pa.string(),
        "rxuxtime": pa.float64(),
        "scaled": pa.bool_(),
        "channel": pa.string(),
        "nmea": pa.list_(pa.string()),
        "signalpower": pa.float64(),
        "ppm": pa.float64(),
        "type": pa.int64(),
        "repeat": pa.int64(),
        "mmsi": pa.int64(),
        "country": pa.string(),
        "country_code": pa.string(),
        "status": pa.int64(),
        "status_text": pa.string(),
        "turn_unscaled": pa.int64(),
        "turn": pa.int64(),
        "speed": pa.float64(),
        "accuracy": pa.bool_(),
        "lon": pa.float64(),
        "lat": pa.float64(),
        "course": pa.float64(),
        "heading": pa.int64(),
        "second": pa.int64(),
        "maneuver": pa.int64(),
        "raim": pa.bool_(),
        "radio": pa.int64(),
        "sync_state": pa.int64(),
        "slot_timeout": pa.int64(),
        "slot_offset": pa.int64(),
        "utc_hour": pa.int64(),
        "utc_minute": pa.int64(),
        "timestamp": pa.timestamp("us", tz="UTC"),
        "msg_type": pa.int64(),
}

SCHEMA_KEYS = list(CANONICAL_SCHEMA.keys())
ARROW_SCHEMA = pa.schema([(k, v) for k, v in CANONICAL_SCHEMA.items()])

In [7]:
def decode_nmea(msg):
        try:
                nmea = msg.get("nmea")
                if not nmea:
                        return msg
                if isinstance(nmea, list):
                        while isinstance(nmea, list) and len(nmea) > 0:
                                nmea = nmea[0]
                if isinstance(nmea, (bytes, bytearray)):
                        nmea = nmea.decode(errors="ignore")
                if not isinstance(nmea, str):
                        return msg

                decoded = pyais.decode(nmea)
                if hasattr(decoded, "__dict__"):
                        decoded_dict = decoded.__dict__
                else:
                        decoded_dict = decoded.asdict() if hasattr(decoded, "asdict") else {}
                for k, v in decoded_dict.items():
                        if hasattr(v, "value"):
                                decoded_dict[k] = v.value
                msg.update(decoded_dict)

        except Exception:
                pass

        return msg

def parse_timestamp(msg):
        if "timestamp" in msg:
                try:
                        dt = datetime.fromisoformat(msg["timestamp"].replace("Z", "+00:00"))
                        return dt
                except:
                        pass
        if "rxuxtime" in msg:
                try:
                        return datetime.fromtimestamp(msg["rxuxtime"], tz=timezone.utc)
                except:
                        pass
        if "rxtime" in msg:
                try:
                        return datetime.strptime(msg["rxtime"], "%Y%m%d%H%M%S").replace(tzinfo=timezone.utc)
                except:
                        pass
        return None

def repair_timestamp(current_dt, previous_dt):
        if current_dt is None:
                if previous_dt:
                        return previous_dt + timedelta(seconds=1)
                return None
        if current_dt.year < 2015:
                if previous_dt:
                        if abs(previous_dt.year - current_dt.year) > 10:
                                return current_dt.replace(year=previous_dt.year)
                return current_dt.replace(year=datetime.now().year)
        if previous_dt:
                delta = (current_dt - previous_dt).total_seconds()
                if delta < -3600:
                        return previous_dt + timedelta(seconds=1)
        return current_dt

def parse_ais(infile, outfile, bsize, start, end):
        if os.path.exists(outfile):
                raise FileExistsError("File already exists")
        
        writer = pq.ParquetWriter(outfile, ARROW_SCHEMA)
        last_seen = {}
        batch = []
        skipped, saved = 0, 0
        with open(infile, "rb") as f:
                for i, line in enumerate(tqdm(f, desc="Parsing AIS")):
                        # if i > 250: # Limit the read
                        #         break
                        try:
                                msg = orjson.loads(line)

                                mmsi = msg.get("mmsi")
                                previous_timestamp = last_seen.get(mmsi)
                                raw_ts = parse_timestamp(msg)
                                corrected_ts = repair_timestamp(raw_ts, previous_timestamp)
                                msg["timestamp"] = corrected_ts if corrected_ts else None
                                if mmsi and corrected_ts:
                                        last_seen[mmsi] = corrected_ts

                                ts = msg["timestamp"]
                                if ts is None:
                                        skipped += 1
                                        continue
                                if start and ts <= start:
                                        skipped += 1
                                        continue
                                if end and ts >= end:
                                        skipped += 1
                                        continue

                                saved += 1
                                msg = decode_nmea(msg)
                                batch.append({k: msg.get(k, None) for k in SCHEMA_KEYS})
                                if len(batch) >= bsize:
                                        df = pa.Table.from_pylist(batch, schema=ARROW_SCHEMA)
                                        writer.write_table(df)
                                        batch.clear()                                
                        except Exception as e:
                                continue
        if batch:
                df = pa.Table.from_pylist(batch, schema=ARROW_SCHEMA)
                writer.write_table(df)

        writer.close()
        return saved, skipped, saved+skipped 

In [8]:
parse_ais(
        infile="dataset/feup.json",
        outfile="dataset/ais_d22-23.parquet",
        bsize=500000,
        start=datetime(2026, 5, 22, 0, 0, 0, tzinfo=timezone.utc),
        end=datetime(2026, 5, 24, 0, 0, 0, tzinfo=timezone.utc)
)

Parsing AIS: 16643643it [01:07, 248367.09it/s]


(428402, 15280259, 15708661)

In [9]:
parse_ais(
        infile="dataset/feup.json",
        outfile="dataset/ais_d31.parquet",
        bsize=500000,
        start=datetime(2026, 5, 31, 0, 0, 0, tzinfo=timezone.utc),
        end=datetime(2026, 6, 1, 0, 0, 0, tzinfo=timezone.utc)
)

Parsing AIS: 16643643it [01:20, 206040.44it/s]


(542211, 15166450, 15708661)

In [10]:
parse_ais(
        infile="dataset/feup.json",
        outfile="dataset/ais_d9-12.parquet",
        bsize=500000,
        start=datetime(2026, 6, 9, 0, 0, 0, tzinfo=timezone.utc),
        end=datetime(2026, 6, 13, 0, 0, 0, tzinfo=timezone.utc)
)

Parsing AIS: 16643643it [02:18, 120363.95it/s]


(1828775, 13879886, 15708661)

### Manage Time Windows

In [19]:
coord_center = {"lat": 41.458747,"lon": -8.842215,"half_side_km": 5.59704}
dlat = coord_center["half_side_km"] / 111.0
dlon = coord_center["half_side_km"] / (111.0 * math.cos(math.radians(coord_center["lat"])))

In [18]:
def mapit(path: str):
        colorscheme = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3","#ff7f00", "#ffff33", "#a65628", "#f781bf", "#999999"]
        print(f"Loading AIS temporal dataset: {path}")
        df = (
                pl.scan_parquet(path)
                .select([
                        "mmsi",
                        "lat",
                        "lon",
                        "timestamp",
                        "speed",
                        "course",
                        "heading",
                        "status_text",
                        "country",
                        "country_code",
                        "signalpower",
                        "accuracy",
                ])
                .collect()
                .drop_nulls(["lat", "lon", "timestamp"])
        )
        if df.height == 0:
                raise ValueError("No valid geospatial AIS points found")

        center_lat = df["lat"].mean()
        center_lon = df["lon"].mean()
        m = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles=None)
        color_map = {}
        def get_color(mmsi):
                if mmsi not in color_map:
                        color_map[mmsi] = colorscheme[len(color_map) % len(colorscheme)]
                return color_map[mmsi]

        features = []
        for rec in df.iter_rows(named=True):
                mmsi = rec["mmsi"]
                lat = rec["lat"]
                lon = rec["lon"]
                ts = rec["timestamp"]

                color = get_color(mmsi)
                iso_time = ts.isoformat().replace("+00:00", "Z")
                popup = (
                        f"<b>MMSI:</b> {mmsi}<br>"
                        f"<b>Speed:</b> {rec.get('speed')} kn<br>"
                        f"<b>Course:</b> {rec.get('course')}°<br>"
                        f"<b>Heading:</b> {rec.get('heading')}°<br>"
                        f"<b>Status:</b> {rec.get('status_text')}<br>"
                        f"<b>Country:</b> {rec.get('country')} ({rec.get('country_code')})<br>"
                        f"<b>Time:</b> {iso_time}"
                )
                features.append({
                        "type": "Feature",
                        "geometry": {
                                "type": "Point",
                                "coordinates": [lon, lat]
                        },
                        "properties": {
                                "time": iso_time,
                                "popup": popup,
                                "icon": "circle",
                                "iconstyle": {
                                        "fillColor": color,
                                        "color": color,
                                        "fillOpacity": 0.6,
                                        "radius": 2.5,
                                        "weight": 1
                                }
                        }
                })

        timeline = TimestampedGeoJson(
                {"type": "FeatureCollection", "features": features},
                period="PT1M",
                duration="PT5M",
                transition_time=200,
                auto_play=False,
                add_last_point=True,
                loop=False,
                max_speed=10        
        )
        folium.TileLayer(
                tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
                attr="&copy; OpenStreetMap contributors &copy; CARTO",
                name="CartoDB Positron",
                max_zoom=19,
                subdomains=["a","b","c","d"],
        ).add_to(m)
        lat0 = coord_center["lat"]; lon0 = coord_center["lon"]
        folium.Marker(
                [lat0, lon0],
                popup="WEC_C4 reference point",
                icon=folium.Icon(color="red", icon="info-sign")
        ).add_to(m)
        folium.Rectangle(
                bounds=[[lat0 - dlat, lon0 - dlon], [lat0 + dlat, lon0 + dlon]],
                color="#e41a1c",
                fill=True,
                fill_opacity=0.1,
                weight=2
        ).add_to(m)
        timeline.add_to(m)
        out = f"output/{path.split('/')[-1].replace('.parquet','')}_timeline_map.html"
        m.save(out)
        return out

In [27]:
for path in ["dataset/ais_d22-23.parquet", "dataset/ais_d31.parquet", "dataset/ais_d9-12.parquet"]:
        filtered = (
                pl.scan_parquet(path)
                .filter(
                        (pl.col("lat") > coord_center["lat"] - dlat) &
                        (pl.col("lat") < coord_center["lat"] + dlat) &
                        (pl.col("lon") > coord_center["lon"] - dlon) &
                        (pl.col("lon") < coord_center["lon"] + dlon)
                )
                .collect()
        )
        base = Path(path).with_suffix("")
        filtered.write_parquet(f"{base}_filtered.parquet")
        filtered.drop(["nmea"]).write_csv(f"{base}_filtered.csv")
        mapit(path)
        mapit(f"{base}_filtered.parquet")

Loading AIS temporal dataset: dataset/ais_d22-23.parquet
Loading AIS temporal dataset: dataset/ais_d22-23_filtered.parquet
Loading AIS temporal dataset: dataset/ais_d31.parquet
Loading AIS temporal dataset: dataset/ais_d31_filtered.parquet
Loading AIS temporal dataset: dataset/ais_d9-12.parquet
Loading AIS temporal dataset: dataset/ais_d9-12_filtered.parquet
